# Re-ID evaluation: OSNet on Market-1501 and MSMT17

Reproduce the standard **same-domain** benchmark numbers for OSNet x1.0. Each model is evaluated on the dataset it was trained for ([torchreid model zoo](https://kaiyangzhou.github.io/deep-person-reid/MODEL_ZOO)). That's the fair comparison against published results.

| Dataset | Size | T4 runtime | Expected (euclidean) |
|---|---|---|---|
| Market-1501 | ~1.3 GB | ~10 min | R1 ~94.2 / mAP ~82.6 |
| MSMT17 | ~4 GB | ~40 min | R1 ~74.9 / mAP ~43.8 |

> **Why not the default checkpoint?** `ReIDModel.from_pretrained()` ships OSNet trained on MSMT17 with `combineall=True` (train+test combined). Evaluating that on MSMT17 leaks the test set (near-100%) and on Market-1501 only measures cross-domain transfer (~61 R1). Below we load each dataset's own same-domain checkpoint instead.

> **Runtime:** T4 GPU (`Runtime → Change runtime type`).

---

## 1. Setup

Install **re-ID** from PyPI (`import reid`). Colab already ships CUDA PyTorch, which pip will reuse when it satisfies the package requirement.


In [ ]:
!pip install -q --upgrade pip
!pip install -q reid matplotlib scikit-learn

In [ ]:
import os
import zipfile
from pathlib import Path

import gdown
import torch

from reid import ReIDModel
from reid.eval import RetrievalProtocol

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
    ROOT = Path("/content")
except ImportError:
    IN_COLAB = False
    ROOT = Path("..").resolve()

CKPT_DIR = ROOT / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | {device}")


def download_checkpoint(gdrive_id: str, filename: str) -> str:
    """Download an OSNet checkpoint from the torchreid model zoo (Google Drive)."""
    path = CKPT_DIR / filename
    if not path.exists():
        gdown.download(id=gdrive_id, output=str(path), quiet=False)
    return str(path)


def print_result(name, metrics, zoo_r1, zoo_map):
    """Print euclidean scores next to the model-zoo targets."""
    print(f"\n{name}")
    print(f"{'metric':<10}{'euclidean':>12}{'model zoo':>12}")
    print("-" * 34)
    print(f"{'Rank-1':<10}{metrics['rank1']:>11.1f}%{zoo_r1:>11.1f}%")
    print(f"{'mAP':<10}{metrics['map']:>11.1f}%{zoo_map:>11.1f}%")
    print(f"{'Rank-5':<10}{metrics['rank5']:>11.1f}%{'-':>12}")
    print(f"{'Rank-10':<10}{metrics['rank10']:>11.1f}%{'-':>12}")
    print(f"{'mINP':<10}{metrics['minp']:>11.1f}%{'-':>12}")

---
## 2. Market-1501

**Expected (OSNet x1.0, Market-trained, model zoo):** R1 ≈ 94.2 / mAP ≈ 82.6

The model-zoo protocol uses **euclidean** distance, so the evaluation below does too.

Market-1501 is a public research benchmark. This notebook downloads the public Google Drive archive linked by the [dataset authors](https://zheng-lab-anu.github.io/Project/project_reid.html) (~1.3 GB extracted); no Roboflow-private data or credentials are used. Review the dataset's terms before use. If Google Drive fails, use the fallback cell below.


In [ ]:
MARKET_ZIP = "/content/Market-1501.zip"
MARKET_DIR = "/content/Market-1501-v15.09.15"

if not os.path.exists(MARKET_DIR):
    # Google Drive file ID for Market-1501-v15.09.15.zip
    gdown.download(id="0B8-rUzbwVRk0c054eEozWG9COHM", output=MARKET_ZIP, quiet=False)
    with zipfile.ZipFile(MARKET_ZIP, "r") as zf:
        zf.extractall("/content")
    print("Extracted to", MARKET_DIR)
else:
    print("Already downloaded.")

In [ ]:
# Fallback if gdown fails: paste your mirror URL:
# !wget -q -O /content/Market-1501.zip "<YOUR_MIRROR_URL>"
# !unzip -q /content/Market-1501.zip -d /content


In [ ]:
from reid import load_market1501

query_m, gallery_m = load_market1501(MARKET_DIR)
print(f"Market-1501: {len(query_m):,} query, {len(gallery_m):,} gallery")

In [ ]:
# OSNet x1.0 checkpoint trained on Market-1501 (model zoo).
market_ckpt = download_checkpoint("1vduhq5DpN2q1g4fYEZfPI17MJeh9qyrA", "osnet_x1_0_market1501.pth")
model_market = ReIDModel.from_pretrained(market_ckpt, architecture="osnet_x1_0")

result_market = model_market.evaluate(
    query_m, gallery_m, protocol=RetrievalProtocol(distance="euclidean"), batch_size=256
)

print_result("Market-1501", result_market.metrics, 94.2, 82.6)

---
## 3. MSMT17

**Expected (OSNet x1.0, MSMT-trained, model zoo):** R1 ≈ 74.9 / mAP ≈ 43.8 (euclidean)

MSMT17 is distributed by Peking University NELVT for academic use only. Request the official **MSMT17_V1** release after signing the [MSMT17 release agreement](https://www.pkuvmc.com/agreement/RELEASE_AGREEMENT-MSMT17.pdf). Do not use unofficial re-uploads until the archive can be verified against the official distribution.

After download, point `MSMT17_DIR` at the extracted root (must contain `test/`, `list_query.txt`, and `list_gallery.txt`).


In [ ]:
# Official MSMT17_V1 only: request from NELVT after signing the release agreement.
# https://www.pkuvmc.com/agreement/RELEASE_AGREEMENT-MSMT17.pdf
MSMT17_DIR = "/content/MSMT17_V1"  # extracted official root

assert os.path.isdir(MSMT17_DIR), (
    f"MSMT17 not found at {MSMT17_DIR}. Request the official MSMT17_V1 release, extract it, then set MSMT17_DIR."
)
assert os.path.isfile(os.path.join(MSMT17_DIR, "list_query.txt")), "Missing list_query.txt"
assert os.path.isfile(os.path.join(MSMT17_DIR, "list_gallery.txt")), "Missing list_gallery.txt"
assert os.path.isdir(os.path.join(MSMT17_DIR, "test")), "Missing test/"
print("Using official MSMT17 at", MSMT17_DIR)

In [ ]:
from reid import load_msmt17

query_ms, gallery_ms = load_msmt17(MSMT17_DIR)
print(f"MSMT17: {len(query_ms):,} query, {len(gallery_ms):,} gallery")

In [ ]:
# Same-domain MSMT17 checkpoint. Skip the default combineall alias (test leakage).
msmt17_ckpt = download_checkpoint("112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M", "osnet_x1_0_msmt17.pth")
model_msmt17 = ReIDModel.from_pretrained(msmt17_ckpt, architecture="osnet_x1_0")

# The full query-by-gallery distance matrix is about 3.8 GB, above the default budget.
result_msmt17 = model_msmt17.evaluate(
    query_ms,
    gallery_ms,
    protocol=RetrievalProtocol(distance="euclidean"),
    batch_size=256,
    max_pairs=1_000_000_000,
)

print_result("MSMT17", result_msmt17.metrics, 74.9, 43.8)

---
## 4. Summary


In [ ]:
print(f"{'Dataset':<14}{'encoder':<22}{'distance':<12}{'mAP':>8}{'Rank-1':>9}{'Rank-5':>9}{'Rank-10':>9}{'mINP':>8}")
print("-" * 91)


def _row(name, encoder, dist, m):
    return (
        f"{name:<14}{encoder:<22}{dist:<12}{m['map']:>7.1f}%{m['rank1']:>8.1f}%"
        f"{m['rank5']:>8.1f}%{m['rank10']:>8.1f}%{m['minp']:>7.1f}%"
    )


print(_row("Market-1501", "OSNet", "euclidean", result_market.metrics))
print(_row("MSMT17", "OSNet", "euclidean", result_msmt17.metrics))
print("-" * 91)
print("Model-zoo targets (OSNet, euclidean): Market-1501 R1≈94.2 mAP≈82.6  |  MSMT17 R1≈74.9 mAP≈43.8")